In [ ]:
pip install mysql-connector-python

In [ ]:
pip install tabulate

In [25]:
# =========================================
# DATABASE CONNECTION
# =========================================

import mysql.connector

def get_connection():
    return mysql.connector.connect(
        host="localhost",
        user="app_user",      # dùng user bạn đã tạo
        password="fullfunction1",
        database="EventDB"
    )
# 👉 Feature:
# - Kết nối Python với MySQL
# - Dùng cho tất cả module phía dưới

In [ ]:
def register_guest(event_id, guest_id):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    INSERT INTO Registrations (EventID, GuestID, RegistrationDate)
    VALUES (%s, %s, NOW())
    """

    try:
        cursor.execute(query, (event_id, guest_id))
        conn.commit()
        print("✅ Registration successful!")
    except Exception as e:
        print("❌ Error:", e)

    cursor.close()
    conn.close()
    
    
#register_guest(2,478)
# 👉 Feature:
# - Đăng ký guest vào event

✅ Registration successful!


In [41]:
def update_registration(registration_id, new_event_id, new_guest_id):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    UPDATE Registrations
    SET EventID = %s,
        GuestID = %s
    WHERE RegistrationID = %s
    """

    cursor.execute(query, (new_event_id, new_guest_id, registration_id))
    conn.commit()

    if cursor.rowcount == 0:
        print("❌ No registration found to update")
    else:
        print(f"✅ Updated successfully! ({cursor.rowcount} row)")

    cursor.close()
    conn.close()

update_registration(1, 2, 4)
# 👉 Feature:
# - Update trạng thái tham gia

✅ Updated successfully! (1 row)


In [ ]:
def update_event(event_id, new_date):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    UPDATE Events
    SET EventDate = %s
    WHERE EventID = %s
    """

    cursor.execute(query, (new_date, event_id))
    conn.commit()

    # 👇 kiểm tra kết quả
    if cursor.rowcount == 0:
        print("❌ Event not found")
    else:
        print(f"✔ Event updated! ({cursor.rowcount} row)")

    cursor.close()
    conn.close()
# update_event(1, '2026-07-01')
# 👉 Feature:
# - Chỉnh sửa lịch event

✔ Event updated! (1 row)


In [58]:
from tabulate import tabulate

def participation_report():
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    SELECT 
        e.EventName,
        COUNT(r.RegistrationID) AS TotalGuests
    FROM Events e
    LEFT JOIN Registrations r ON e.EventID = r.EventID
    GROUP BY e.EventID
    """

    cursor.execute(query)
    results = cursor.fetchall()

    headers = ["Event Name", "Total Guests"]

    print("\n📊 Participation Report:")
    print(tabulate(results, headers=headers, tablefmt="grid"))

    cursor.close()
    conn.close()

participation_report()
# 👉 Feature:
# - Xem event có bao nhiêu người


📊 Participation Report:
+------------------+----------------+
| Event Name       |   Total Guests |
+==================+================+
| Tech Conference  |             41 |
+------------------+----------------+
| Music Festival   |             46 |
+------------------+----------------+
| Startup Meetup   |             59 |
+------------------+----------------+
| AI Workshop      |             53 |
+------------------+----------------+
| Business Forum   |             49 |
+------------------+----------------+
| Art Exhibition   |             56 |
+------------------+----------------+
| Gaming Event     |             61 |
+------------------+----------------+
| Education Fair   |             47 |
+------------------+----------------+
| Health Seminar   |             45 |
+------------------+----------------+
| Marketing Summit |             55 |
+------------------+----------------+


In [51]:
def guest_statistics():
    conn = get_connection()
    cursor = conn.cursor()

    # 👉 tổng số khách
    cursor.execute("SELECT COUNT(*) FROM Guests")
    total_guests = cursor.fetchone()[0]

    # 👉 số người đã đăng ký
    cursor.execute("SELECT COUNT(DISTINCT GuestID) FROM Registrations")
    participated = cursor.fetchone()[0]

    # 👉 tính %
    if total_guests == 0:
        rate = 0
    else:
        rate = (participated / total_guests) * 100

    print("\n📊 Guest Statistics:")
    print(f"👥 Total Guests: {total_guests}")
    print(f"✅ Participated: {participated}")
    print(f"📈 Participation Rate: {rate:.2f}%")

    cursor.close()
    conn.close()

guest_statistics()
# 👉 Feature:
# - Báo cáo % tham gia thực tế


📊 Guest Statistics:
👥 Total Guests: 510
✅ Participated: 324
📈 Participation Rate: 63.53%


In [60]:
def check_schedule_conflict(date):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    SELECT EventName, EventDate
    FROM Events
    WHERE EventDate = %s
    """

    cursor.execute(query, (date,))
    results = cursor.fetchall()

    print(f"\n📅 Checking date: {date}")

    if len(results) == 0:
        print("❌ No events on this date")

    elif len(results) == 1:
        print("✅ No conflict (only 1 event)")
        print(f"• {results[0][0]}")

    else:
        print("⚠️ Conflict detected! Multiple events:")
        for row in results:
            print(f"• {row[0]}")

    cursor.close()
    conn.close()
    
check_schedule_conflict('2026-07-01')
# 👉 Feature:
# - Phát hiện event trùng lịch


📅 Checking date: 2026-07-01
⚠️ Conflict detected! Multiple events:
• Tech Conference
• Gaming Event


In [61]:
# =========================================
# SIMPLE CLI MENU
# =========================================

def main():
    while True:
        print("\n=== EVENT MANAGEMENT SYSTEM ===")
        print("1. Register Guest")
        print("2. Confirm Attendance")
        print("3. Update Event")
        print("4. View Participation Report")
        print("5. Guest Statistics")
        print("6. Check Schedule Conflict")
        print("0. Exit")

        choice = input("Choose: ")

        if choice == "1":
            event_id = int(input("Event ID: "))
            guest_id = int(input("Guest ID: "))
            register_guest(event_id, guest_id)

        elif choice == "2":
            reg_id = int(input("Registration ID: "))
            confirm_attendance(reg_id)

        elif choice == "3":
            event_id = int(input("Event ID: "))
            name = input("New Event Name: ")
            update_event(event_id, name)

        elif choice == "4":
            participation_report()

        elif choice == "5":
            guest_statistics()

        elif choice == "6":
            date = input("Enter date (YYYY-MM-DD): ")
            check_schedule_conflict(date)

        elif choice == "0":
            break

        else:
            print("❌ Invalid choice")


if __name__ == "__main__":
    main()


=== EVENT MANAGEMENT SYSTEM ===
1. Register Guest
2. Confirm Attendance
3. Update Event
4. View Participation Report
5. Guest Statistics
6. Check Schedule Conflict
0. Exit

📅 Checking date: 2026-07-01
⚠️ Conflict detected! Multiple events:
• Tech Conference
• Gaming Event

=== EVENT MANAGEMENT SYSTEM ===
1. Register Guest
2. Confirm Attendance
3. Update Event
4. View Participation Report
5. Guest Statistics
6. Check Schedule Conflict
0. Exit
